In [2]:
import pandas as pd
data = pd.read_csv("../data/processed/model_data.csv", parse_dates=["cutoff"])
print(data.shape)   # (22182, 13)

(22182, 13)


Loaded 22,182 customer snapshots (5 cutoffs) saved by notebook 04.

In [4]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

FEATURES = ["recency_days", "tenure_days", "frequency", "monetary", "avg_order_value",
            "purchases_last_90d", "revenue_last_90d",
            "purchases_last_180d", "revenue_last_180d", "avg_gap_days"]

def evaluate(name, y, score):
    base = y.mean()
    k = int(round(0.10 * len(y)))
    top = np.argsort(-score, kind="stable")[:k]
    prec_top = y[top].mean()
    return {"model": name,
            "PR-AUC": round(average_precision_score(y, score), 3),
            "ROC-AUC": round(roc_auc_score(y, score), 3),
            "top10%_precision": round(prec_top, 3),
            "top10%_lift": round(prec_top / base, 2),
            "max_possible_lift": round(1 / base, 2)}

def rfm_score(df):
    return ((-df["recency_days"]).rank(pct=True)
            + df["frequency"].rank(pct=True)
            + df["monetary"].rank(pct=True)).values

def calib_table(y, p):
    d = pd.DataFrame({"p": p, "y": y})
    d["bin"] = pd.qcut(d["p"], 10, duplicates="drop")
    return d.groupby("bin", observed=True).agg(avg_predicted=("p", "mean"),
                                               actual_rate=("y", "mean"),
                                               customers=("y", "size")).round(3)

def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p)).reshape(-1, 1)

train = data[data["cutoff"].isin(pd.to_datetime(["2010-09-01", "2010-12-01", "2011-03-01"]))]
val = data[data["cutoff"] == pd.Timestamp("2011-06-01")]
y_val = val["label"].values
print(len(train), len(val))    # expect 12050 4908

12050 4908


Train: Sep 2010, Dec 2010, Mar 2011 cutoffs (12,050 rows). Validation: Jun 2011 (4,908). Test: Sep 2011, used once.

In [5]:
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                    max_iter=200, random_state=42)
gb.fit(train[FEATURES], train["label"])
p_val = gb.predict_proba(val[FEATURES])[:, 1]

platt = LogisticRegression(C=1e6).fit(logit(p_val), y_val)   # effectively unregularised
p_val_cal = platt.predict_proba(logit(p_val))[:, 1]

print("Platt slope/intercept:", platt.coef_.round(3), platt.intercept_.round(3))
print("validation mean: raw", round(p_val.mean(), 3),
      "| calibrated", round(p_val_cal.mean(), 3), "| actual", round(y_val.mean(), 3))
print("Brier: raw", round(brier_score_loss(y_val, p_val), 4),
      "| calibrated", round(brier_score_loss(y_val, p_val_cal), 4))

Platt slope/intercept: [[1.19]] [0.051]
validation mean: raw 0.34 | calibrated 0.329 | actual 0.329
Brier: raw 0.1539 | calibrated 0.1531


On validation, gradient boosting over-predicts slightly (mean 0.340 vs 0.329 actual). Platt scaling (slope 1.19, intercept 0.051) fixes this on the same data, so it is only a sanity check.

In [6]:
test = data[data["cutoff"] == pd.Timestamp("2011-09-01")]
y_test = test["label"].values
p_test = gb.predict_proba(test[FEATURES])[:, 1]
p_test_cal = platt.predict_proba(logit(p_test))[:, 1]

print("test customers:", len(test), "| base rate:", round(y_test.mean(), 3))
print(pd.DataFrame([evaluate("RFM rank score", y_test, rfm_score(test)),
                    evaluate("Gradient boosting", y_test, p_test)]).to_string(index=False))

print("\nmean predicted: raw", round(p_test.mean(), 3),
      "| calibrated", round(p_test_cal.mean(), 3), "| actual", round(y_test.mean(), 3))
print("Brier: raw", round(brier_score_loss(y_test, p_test), 4),
      "| calibrated", round(brier_score_loss(y_test, p_test_cal), 4))
print("\nRaw"); print(calib_table(y_test, p_test).to_string())
print("\nCalibrated"); print(calib_table(y_test, p_test_cal).to_string())

test customers: 5224 | base rate: 0.428
            model  PR-AUC  ROC-AUC  top10%_precision  top10%_lift  max_possible_lift
   RFM rank score   0.749    0.785             0.898         2.10               2.34
Gradient boosting   0.767    0.793             0.937         2.19               2.34

mean predicted: raw 0.323 | calibrated 0.312 | actual 0.428
Brier: raw 0.1915 | calibrated 0.1943

Raw
                 avg_predicted  actual_rate  customers
bin                                                   
(0.039, 0.0637]          0.054        0.088        523
(0.0637, 0.123]          0.096        0.160        524
(0.123, 0.156]           0.138        0.247        527
(0.156, 0.189]           0.173        0.291        516
(0.189, 0.225]           0.206        0.308        522
(0.225, 0.293]           0.257        0.391        522
(0.293, 0.419]           0.354        0.497        523
(0.419, 0.524]           0.476        0.605        522
(0.524, 0.727]           0.610        0.757        

Test base rate 0.428. Gradient boosting PR-AUC 0.767 vs 0.749 for the RFM rule; top-10% lift 2.19 of a possible 2.34. Probabilities are too low in every bin (by about 6 to 14 points) and Platt scaling did not help (Brier 0.1915 to 0.1943). Ranking transfers across seasons; absolute probabilities do not. A likely cause is the autumn season, which I have not tested.

In [7]:
import os
os.makedirs("../results", exist_ok=True)

test_results = pd.DataFrame([evaluate("RFM rank score", y_test, rfm_score(test)),
                             evaluate("Gradient boosting", y_test, p_test)])
test_results.insert(0, "cutoff", "2011-09-01")
test_results["base_rate"] = round(y_test.mean(), 3)
test_results.to_csv("../results/test_performance.csv", index=False)

calib_raw = calib_table(y_test, p_test).reset_index()
calib_raw["bin"] = calib_raw["bin"].astype(str)
calib_raw.to_csv("../results/test_calibration_raw.csv", index=False)

print(test_results.to_string(index=False))
print(calib_raw.to_string(index=False))

    cutoff             model  PR-AUC  ROC-AUC  top10%_precision  top10%_lift  max_possible_lift  base_rate
2011-09-01    RFM rank score   0.749    0.785             0.898         2.10               2.34      0.428
2011-09-01 Gradient boosting   0.767    0.793             0.937         2.19               2.34      0.428
            bin  avg_predicted  actual_rate  customers
(0.039, 0.0637]          0.054        0.088        523
(0.0637, 0.123]          0.096        0.160        524
 (0.123, 0.156]          0.138        0.247        527
 (0.156, 0.189]          0.173        0.291        516
 (0.189, 0.225]          0.206        0.308        522
 (0.225, 0.293]          0.257        0.391        522
 (0.293, 0.419]          0.354        0.497        523
 (0.419, 0.524]          0.476        0.605        522
 (0.524, 0.727]          0.610        0.757        522
 (0.727, 0.993]          0.862        0.937        523
